# How LLM Application Evaluation Works
*The end-to-end application eval workflow, a worked email-routing example, why RAG systems need retriever-, generator-, and workflow-level evals, and the three risk categories every eval pipeline has to cover*

## From what to how

Defining LLM evals and separating model evals from application evals answers *why* and *what*. The remaining question is *how* — the actual workflow for evaluating a typical LLM-based application, walked through end to end. Since application evals are the ones a builder actually runs day to day, that's the perspective used throughout: evaluating a full system, not benchmarking a raw model.

## The application evaluation workflow

The same workflow applies whether the system under test is a five-minute classification script or a complex multi-step agent. It has the following stages:

1. Define the task and the target — what exactly is being evaluated.
2. Define a success criterion and the metric that measures it.
3. Build a dataset — typically called a **golden dataset** in this context.
4. Choose an evaluation method — automated, human, or another LLM acting as a judge.
5. Run the system against the dataset.
6. Evaluate the results against the metric.
7. Analyze where and why the system is failing.
8. Improve the system — usually the prompt or the underlying model.
9. Iterate: rerun the evaluation, and repeat steps 5–8 until performance is acceptable.
10. Deploy.
11. Monitor in production, and feed real failures back into the golden dataset.

That last step closes the loop — production failures become new evaluation data, which starts the cycle again. This is also what makes evals **repeatable**: the same golden dataset gets used every time the system changes, so results across versions are directly comparable.

```mermaid
flowchart TD
    A["Define task<br/>and target"] --> B["Define success<br/>criteria and metric"]
    B --> C["Build a<br/>golden dataset"]
    C --> D["Choose an<br/>evaluation method"]
    D --> E["Run the system<br/>on the dataset"]
    E --> F["Evaluate<br/>the results"]
    F --> G["Analyze where<br/>it's failing"]
    G --> H["Improve the system<br/>(prompt or model)"]
    H --> E
    F --> I{"Good<br/>enough?"}
    I -->|No| G
    I -->|Yes| J["Deploy"]
    J --> K["Monitor in<br/>production"]
    K --> L["Add failures to the<br/>golden dataset"]
    L --> C
```

## A worked example: routing support emails

Consider a large company that receives a high volume of customer support emails every day and wants to automate the first step of handling them: reading each email and classifying it as a **billing** issue, a **technical** issue, or a **general** query, then routing it to the right team. Without this, someone has to read and tag every email manually.

The system built to solve this is intentionally simple: an LLM given a prompt that describes its job — read the email, decide which category it belongs to, and output that category. A system this simple can be built in a matter of minutes. The real question is whether it can be deployed as-is, and the answer is no — it needs to be evaluated first, using the same workflow above.

## Applying the workflow: task, target, and success criteria

**Task and target.** The target is the routing system as a whole; the task is checking whether it classifies emails correctly. This makes it a classification problem.

**Success criterion and metric.** For a classification task, the natural metric is accuracy: out of 100 incoming emails, how many get routed to the correct team? A system that routes 90 out of 100 correctly is 90% accurate. Simple, single-number metrics like this are typical for classification-style evaluations.

## Building the golden dataset

The dataset pairs each email's content with the category it should have been assigned, decided manually ahead of time. A few examples:

| Email content | Correct category |
|---|---|
| "My card was charged twice" | Billing |
| "The app crashes on login" | Technical |
| "What are your support hours?" | General |

A real dataset like this typically runs from 50 to 500 rows, not just a handful. The strongest source for it is actual historical data — past support conversations pulled from the real system, manually labeled by a person, rather than invented examples. This labeled dataset is what gets reused every time the system is evaluated again, which is what makes the process repeatable.

## Choosing an evaluation method

Once the dataset exists, something has to decide whether each of the system's answers is correct. There are three options:

- **Automated.** For a task with a small, fixed set of possible outputs — like "billing / technical / general" — a script can compare the system's output to the labeled answer directly. No human or LLM judgment is needed, and it's the cheapest, fastest option.
- **Human.** For open-ended output — a long, freeform text answer from a chatbot, say — comparing two paragraphs for correctness isn't something a simple script can do reliably, since two answers can be worded completely differently and still mean the same thing. A human reviewer can judge that, but doing so at scale is slow and costly.
- **LLM-as-judge.** A middle ground: another LLM is used to compare the system's output against the expected answer and judge whether they match in meaning, at a fraction of the cost and time of human review.

For the email classification example, the output space is small and fixed, so an automated script comparing predicted labels to the golden dataset is the right choice. A chatbot producing long freeform answers would typically call for human review or an LLM-as-judge instead.

## Running, evaluating, and analyzing

**Run the system.** The golden dataset's inputs are sent through the system, and it produces a category for each one.

**Evaluate the results.** The predicted categories are compared against the correct ones, and an accuracy score comes out — say, 80 out of 100 emails classified correctly, for 80% accuracy.

**Analyze the results.** An 80% score on its own doesn't say what to fix. Analysis means looking at which emails were misclassified and why. For a text classification system like this one, two likely causes stand out: the system prompt may be defined in a way that causes confusion — for example, between billing and technical issues — or the underlying model itself may not be capable enough for the task, particularly if it's a smaller or lower-capacity model.

## Improving and iterating

Based on that analysis, there are two levers to pull: rewrite the system prompt to remove the ambiguity causing misclassifications, or swap in a stronger underlying model. Either change targets the *system*, not the model in isolation — the goal is a better-performing application, not a "better" model for its own sake.

After a change, the same golden dataset is run again and a new accuracy score comes out. Suppose tightening the prompt raises accuracy from 80% to 90%; if that's still not sufficient, switching to a more capable model might push it to 95%. This is the iteration loop: adjust, rerun the same evaluation, check the new score, and repeat until performance clears whatever bar makes the system worth shipping.

## Deployment doesn't end the loop

Once accuracy is high enough, the system gets deployed — but evaluation doesn't stop there. Production traffic is monitored continuously, because a system that scored 95% on its test dataset can still make mistakes on real customer emails it has never seen before.

When a production failure surfaces — an email that should have been routed to billing but was sent to the technical team, say — that specific example gets pulled and added directly into the golden dataset. The evaluation workflow then restarts from that point: the dataset is now richer, and the system is evaluated and improved against it again. Every real-world failure becomes new training signal for the next round of evaluation, and the golden dataset keeps growing more representative of what the system actually encounters in production. This loop keeps running for as long as the system stays deployed.

## One application, several evals

A single LLM-based application commonly has more than one eval running against it, not just one. The email-classification example above involved a single evaluation, but a more complex system typically doesn't stop there. A RAG application, for instance, might run separate evals for:

- The **retriever's** performance on its own.
- The **embedding model's** quality.
- The **full RAG pipeline**, end to end.
- The system's **latency**.

Each of these targets a different component or a different concern, and each has its own dataset, criteria, and metric. It's worth holding onto this as a general pattern: one application, several evals, each answering a different question about how well that application actually works.

## Why multiple eval pipelines: a RAG example

A single evaluation pipeline usually isn't enough to cover a whole application, because an LLM-based system typically has more than one place where it can fail.

Take a RAG chatbot as a concrete case. A basic RAG setup has two core components: a **retriever**, connected to a vector database, and a **generator** — an LLM that turns retrieved context into an answer. A query arrives, the retriever pulls relevant documents from the vector database based on that query, and both the query and the retrieved documents are passed to the generator, which produces the final answer grounded in that context.

This structure has (at least) two independent failure points:

- The **retriever** can fetch the wrong documents — irrelevant or incomplete ones — which sets the generator up to produce a wrong answer even if the generator itself is working correctly.
- The **generator** can fail on its own, ignoring perfectly good retrieved context and hallucinating an answer that isn't grounded in it.

Because these are two separate places where things can go wrong, each needs its own evaluation pipeline. A single pipeline covering the whole RAG system end to end wouldn't tell you *which* of these two components caused a bad answer — only that something did.

## Evaluating the retriever

The retriever's job is simple to state: given a query, pull the documents from the vector database that are actually relevant to it. An evaluation pipeline built for the retriever checks exactly that — given a query, are the right, relevant documents being retrieved? This pipeline is scoped to the retriever alone; it doesn't concern itself with what the generator eventually does with those documents.

## Evaluating the generator: faithfulness

The generator's job is also simple to state: given a context, produce an answer grounded in it. An evaluation pipeline for the generator checks **faithfulness** — sometimes called **groundedness** — meaning the answer should come entirely from the given context, without inventing additional facts that weren't in it.

For example, suppose the query is "What is the duration of the machine learning course?" and the retrieved document says the duration is three weeks. A faithful answer states exactly that — three weeks — and nothing more. An unfaithful answer would add unrelated details the model generated on its own, like recommending a different course or inventing that course's duration, none of which came from the retrieved context. The generator eval pipeline exists specifically to catch that gap between what the context actually says and what the generated answer claims.

## Does a working retriever and generator guarantee a working application?

Suppose both pipelines above are in place, and both report that the retriever and the generator are individually performing well. That raises a natural question: does that combination guarantee the RAG application as a whole works correctly?

It's a question worth sitting with before assuming the answer is yes — component-level evaluation and system-level evaluation aren't automatically the same thing, and where they can diverge is worth examining directly.

## A worked example: correct components, broken pipeline

Consider a query — "What is the duration of the machine learning course?" — sent to a retriever configured with `k = 5`, meaning it pulls back the five most relevant documents from the vector database. Suppose four of those five are unrelated, and the fifth, ranked last, states plainly: "The duration of the ML course is 8 weeks."

Judged purely on its own goal — surface the correct document somewhere within the top *k* results — the retriever did its job. The right document was in there.

All five documents, plus the original query, get passed to the generator. Suppose the generator's system prompt instructs it to weigh the higher-ranked documents more heavily when constructing an answer — a reasonable instruction on its own. But because the correct document happened to be ranked last, the generator leans on the higher-priority documents instead, one of which mentions a *different* course — say, a Python course with an 8-week duration — and produces the answer: "The duration of the ML course is 8 weeks," attributing the wrong course's timeline to the one that was actually asked about.

That answer is wrong. But notice what didn't happen: the generator didn't hallucinate a number out of nowhere, and it didn't ignore its instructions — it faithfully prioritized the higher-ranked context, exactly as it was told to. Both the retriever and the generator were independently doing what they were built to do. The application still gave the wrong answer, because the *combination* of the two broke down — the correct document existed, but its low rank meant the generator effectively never saw it as a priority.

## The fix: a workflow-level eval

This is exactly the gap that component-level evals miss. A retriever eval alone would report success — the right document was fetched. A generator eval alone might also report success, since the generator followed its instructions faithfully given what it was handed. Neither eval, run in isolation, would catch this failure.

Catching it requires a third pipeline: a **workflow-level eval** that evaluates the retriever and generator *together*, checking whether their combination produces the right final answer — not whether each one is individually well-behaved. Running that eval on this example would flag the wrong answer and point to the actual cause: the correct document was buried at the bottom of the ranking instead of near the top.

The fix that follows directly from that diagnosis is a **reranker** — a component that reorders retrieved documents after the initial retrieval step, based on relevance to the query, so the most useful document moves to the top instead of sitting last. Adding one here would push the correct document to the top of the ranking, and the pipeline would start producing the right answer.

## Workflow-level success still isn't the final word

Suppose the retriever eval, the generator eval, and the new workflow-level eval are all now passing — the right documents are retrieved, the generator produces faithful answers, and the combination of the two works correctly together. Does that guarantee the application is ready for real users?

Not necessarily. Suppose the entire pipeline, working correctly end to end, takes 10 seconds to answer a single query — the user types a question and waits 10 seconds for a response. Every component behaved correctly, and the workflow behaved correctly, but a 10-second response time still isn't acceptable for production. Catching that requires yet another eval, this one scoped to the **application as a whole**: checking that latency stays under an acceptable threshold, regardless of whether every individual piece underneath it is functioning correctly.

## Three levels where failure points exist

Put together, failure in an LLM-based application can show up at three distinct levels, and each one needs its own evaluation:

- **Component level.** Any individual piece can fail on its own — a system prompt, a retriever, a reranker, a query rewriter, an embedding model, a vector database, an output parser, a tool selector, memory, or guardrails, depending on what the application is built from. Each of these gets its own eval pipeline.
- **Workflow level.** Even when every component passes its own eval, the way they interact can still break — as in the retriever-and-generator example above. This applies just as much to an agent's workflow or a multi-turn chatbot's conversation flow as it does to RAG.
- **Application level.** Even when the workflow itself works correctly, whole-system concerns like latency, token cost per query, and time to first token still need to be checked directly, since none of the lower-level evals are set up to catch them.

A pipeline that only tests components, or only tests the workflow, or only tests the top-level application, will each miss failures the others are built to catch. That's the underlying reason a single LLM-based application ends up needing several evals rather than one.

```mermaid
flowchart TD
    A["Failure points in an<br/>LLM-based application"] --> B["Component level"]
    A --> C["Workflow level"]
    A --> D["Application level"]

    B --> B1["retriever, reranker,<br/>embedding model,<br/>system prompt,<br/>output parser..."]
    C --> C1["how components<br/>work together<br/>(e.g. retriever + generator)"]
    D --> D1["latency, token cost,<br/>time to first token"]
```

## A second reason for multiple evals: risk categories

Multiple failure points is one reason a single LLM-based application ends up needing several eval pipelines. There's a second, independent reason: **risk categories**.

Evals can be placed on three things — individual components, workflows, and the entire application — but within each of those, there are multiple distinct *aspects* worth checking, not just one. Take a RAG chatbot as an example again:

- At the **application level**, correctness and helpfulness obviously matter — but so does **safety**. An answer can be entirely correct and still be a failure if, say, it leaks another user's phone number or email address in the process.
- At the **workflow level** — the retriever-and-generator combination discussed earlier — groundedness and faithfulness are one concern, but **cost** is a separate one: the same correct, faithful answer is a problem if producing it costs far more than it should.
- At the **component level**, a retriever can be fetching entirely correct documents and still be a problem if it takes 5 or 10 seconds to do so — **latency** is its own dimension, independent of correctness.

So it's not just that there are multiple failure points; each failure point also has multiple risk dimensions attached to it. Those dimensions are organized into three broad **risk categories**: application quality, safety, and operations.

```mermaid
flowchart TD
    A[Risk categories] --> B["Application quality"]
    A --> C["Safety"]
    A --> D["Operations"]
    B --> B1["Does the app do<br/>its job well?"]
    C --> C1["Is the output<br/>free of harm?"]
    D --> D1["Does it run fast,<br/>cheap, and reliably?"]
```

## Application quality

Application quality covers whether the system does its actual job well — whether it gives correct, relevant, complete answers to what the user asked. This category looks different depending on what kind of application is being evaluated.

**General LLM applications** — a text summarizer, for instance — are typically checked against:

- **Correctness and accuracy** — is the generated content factually right?
- **Relevance** — does the answer actually address what was asked?
- **Completeness** — were all parts of the question answered?
- **Instruction following** — if a specific format or length was requested, was it honored?

### RAG applications

RAG applications add a further set of quality dimensions specific to retrieval and grounding:

- **Context relevance** — are the retrieved documents actually relevant to the query?
- **Retriever recall** — closely related to context relevance, checking whether the right documents are being found at all.
- **Groundedness and faithfulness** — is the generated answer based entirely on the retrieved context, without inventing extra information?
- **Citation accuracy** — when an answer cites a specific source for a specific claim, does that citation actually point to where the claim came from?

### Agent-based applications

Agent-based applications introduce a different set of quality concerns, tied to multi-step decision-making and tool use:

- **Tool selection** — is the agent choosing the right tool for the task at hand?
- **Parameter correctness** — when a tool is called, are the correct parameters being passed to it?
- **Task completion** — is the agent actually completing the task, or failing at a high rate?
- **Error recovery** — if something goes wrong mid-task, can the agent recover from it rather than getting stuck or compounding the error?

### Multi-turn chatbots

Where a conversation continues across several exchanges, two more dimensions apply:

- **Context retention** — how much of the earlier conversation does the chatbot actually remember and use?
- **Clarification behavior** — when the user's request is ambiguous or unclear, does the chatbot ask a clarifying question instead of guessing?

## Safety

Safety covers whether the output is free of harm, independent of whether it's otherwise correct. It typically spans:

- **Toxicity** — is the output free of toxic language?
- **Harmful content** — does it avoid content related to self-harm, weapons, or illegal activity?
- **Bias** — does the system respond consistently, rather than answering differently depending on assumptions about who's asking?
- **Privacy leaks** — does it avoid exposing personal information it shouldn't, such as someone's credit card number or contact details?
- **Prompt injection and jailbreak resistance** — can the system be manipulated, through crafted prompts, into doing something it was explicitly built not to do?

## Operations

Operations covers whether the system runs fast, cheap, and reliably once deployed:

- **Latency** — how long a response takes.
- **Cost per request** — what each query costs to serve.
- **Token efficiency** — how economically the system uses tokens to produce its answer.
- **Error and failure rate** — how often requests fail outright.
- **Latency under load** — whether response time holds up once real traffic, not just test traffic, hits the system.

## Why this reinforces the need for multiple pipelines

Each of these risk categories typically gets its own evaluation pipeline, even within a single application: one pipeline for latency, a separate one for safety, another for correctness, and so on. Combined with the earlier point about multiple failure points — components, workflows, and the application as a whole — this is why, in practice, almost any LLM-based application ends up with more than one evaluation pipeline running against it rather than a single, all-purpose eval. The two reasons are independent of each other, but they compound: more failure points multiplied by more risk dimensions per failure point adds up to a genuinely multi-pipeline evaluation setup for anything beyond the simplest system.